# 1. Konfiguracja środowiska oraz datasetu

## 1.1. Instalacja zależności

In [ ]:
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#%pip install -r requirements.txt

## 1.2.Konfiguracja importów

In [ ]:
import torch
import random
from pathlib import Path
from IPython.display import Audio
from src.process_guitarset import process_dataset
from src.config import *
from src.dataset import GuitarSetDataset, collate_fn
from torch.utils.data import DataLoader
from src.visualization import (
    plot_audio_waveform,
    plot_cqt,
    plot_annotation_map,
    create_annotation_maps
)

## 1.3. Konfiguracja GPU - automatyczne wykrywanie

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używane urządzenie: {device}")
if device.type == 'cuda':
    print(f"Model karty GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

## 1.4. Wczytanie guitarsetu

In [ ]:
config = {
    "data_dir": "src/guitarset_data",
    "output_dir": "src/processed_data",
    "seed": 42,
    "apply_augmentation": False,
    "max_tracks": None,
    "overwrite": False
}

process_dataset(**config)

## 1.5. Wczytanie guitarseta z plików batch

In [ ]:
# Wczytanie danych z plików
train_dir = Path(config["output_dir"]) / "train"
val_dir = Path(config["output_dir"]) / "val"
test_dir = Path(config["output_dir"]) / "test"

train_dataset = GuitarSetDataset(train_dir)
val_dataset = GuitarSetDataset(val_dir)
test_dataset = GuitarSetDataset(test_dir)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4
)

# Przykład: sprawdzanie notes, onsets, contours w train_loader
batch = next(iter(train_loader))

notes = batch['notes']    # [B, T, F_notes]
onsets = batch['onsets']  # [B, T, F_notes]
contours = batch['contours']  # [B, T, F_contours]

# Policzenie ile pozytywnych wartości (czyli > 0)
n_notes_pos = (notes > 0).sum().item()
n_onsets_pos = (onsets > 0).sum().item()
n_contours_pos = (contours > 0).sum().item()

# I jaki jest udział pozytywów w całości
total_notes = notes.numel()
total_onsets = onsets.numel()
total_contours = contours.numel()

print(f"Notes positives: {n_notes_pos} / {total_notes} ({n_notes_pos/total_notes:.6f})")
print(f"Onsets positives: {n_onsets_pos} / {total_onsets} ({n_onsets_pos/total_onsets:.6f})")
print(f"Contours positives: {n_contours_pos} / {total_contours} ({n_contours_pos/total_contours:.6f})")



## 1.6. Wizualizacja przykładowego pliku

In [ ]:
sample = random.choice(train_dataset)

# Dane surowe
audio = sample["audio"].numpy()
cqt = sample["features"].numpy()
sr = sample["sample_rate"]

# Wizualizacja audio i CQT
plot_audio_waveform(audio, sr)
plot_cqt(cqt, sr=AUDIO_SAMPLE_RATE, hop_length=FFT_HOP,
         freq_bins=FREQ_BINS_NOTES)

# Wizualizacja adnotacji nut i konturów
notes_map, contours_map = create_annotation_maps(sample)
plot_annotation_map(notes_map, title="Adnotacje nut (sparse piano roll)", ylabel="Bin częstotliwości (nuty)",
                    cmap='hot')
plot_annotation_map(contours_map, title="Adnotacje konturów (multif0)", ylabel="Bin częstotliwości (kontury)",
                    cmap='Blues')
# Odtwarzanie audio
Audio(audio, rate=sr)


# 2. Model i uczenie

## 2.1. Wczytanie implementacji modelu

In [ ]:
sample = next(iter(train_loader))
input_features = sample["features"].shape[-1]

## 2.2. Pętla treningowa